# Análise Exploratória: Inflação e Poder de Compra no Brasil (2020-Hoje)

Este notebook explora os dados consolidados no banco relacional SQLite `data/inflacao_brasil.db`.

### Perguntas de Negócio:
1. Quais grupos e subitens do IPCA mais subiram no período?
2. Quais subiram acima do reajuste do salário mínimo (+56,02%)?
3. Qual a diferença de peso entre o IPCA (população geral) e o INPC (baixa renda)?

In [ ]:
import os
import sqlite3
import pandas as pd
import plotly.express as px

caminho_db = os.path.join('..', 'data', 'inflacao_brasil.db')
con = sqlite3.connect(caminho_db)
print('Conexão estabelecida com sucesso!')

## 1. Salário Mínimo: Reajuste Nominal Acumulado

In [ ]:
query_sm = """
SELECT 
    (SELECT valor FROM salario_minimo WHERE data = '2020-01-01') AS sm_inicial,
    (SELECT valor FROM salario_minimo WHERE data = (SELECT MAX(data) FROM ipca)) AS sm_final,
    ROUND((( (SELECT valor FROM salario_minimo WHERE data = (SELECT MAX(data) FROM ipca)) / (SELECT valor FROM salario_minimo WHERE data = '2020-01-01') ) - 1.0) * 100.0, 2) AS reajuste_sm_pct;
"""
df_sm = pd.read_sql_query(query_sm, con)
df_sm

## 2. Inflação Acumulada Composta por Grupo de Consumo

In [ ]:
query_grupos = """
SELECT 
    codigo_hierarquico, 
    nome_categoria, 
    ROUND(AVG(peso_mensal), 2) AS peso_medio_pct,
    ROUND((EXP(SUM(LN(1.0 + (variacao_mensal / 100.0)))) - 1.0) * 100.0, 2) AS inflacao_acumulada_pct
FROM ipca
WHERE nivel = 'grupo'
GROUP BY codigo_hierarquico, nome_categoria
ORDER BY inflacao_acumulada_pct DESC;
"""
df_grupos = pd.read_sql_query(query_grupos, con)
df_grupos

## 3. Top 15 Subitens que Mais Subiram vs Salário Mínimo

In [ ]:
query_top = """
WITH acumulado_subitens AS (
    SELECT 
        codigo_hierarquico,
        nome_categoria,
        ROUND(AVG(peso_mensal), 4) AS peso_medio_pct,
        ROUND((EXP(SUM(LN(1.0 + (variacao_mensal / 100.0)))) - 1.0) * 100.0, 2) AS inflacao_acumulada_pct
    FROM ipca
    WHERE nivel = 'subitem'
      AND variacao_mensal IS NOT NULL
      AND (1.0 + (variacao_mensal / 100.0)) > 0
    GROUP BY codigo_hierarquico, nome_categoria
)
SELECT 
    RANK() OVER (ORDER BY inflacao_acumulada_pct DESC) AS ranking,
    nome_categoria,
    inflacao_acumulada_pct,
    peso_medio_pct
FROM acumulado_subitens
ORDER BY ranking ASC
LIMIT 15;
"""
df_top = pd.read_sql_query(query_top, con)
df_top

## 4. Comparação de Pesos no Orçamento: IPCA vs INPC

In [ ]:
query_pesos = """
SELECT 
    ipca.nome_categoria,
    ROUND(AVG(ipca.peso_mensal), 2) AS peso_ipca,
    ROUND(AVG(inpc.peso_mensal), 2) AS peso_inpc,
    ROUND(AVG(inpc.peso_mensal) - AVG(ipca.peso_mensal), 2) AS diferenca_pp
FROM ipca
JOIN inpc 
  ON ipca.data = inpc.data 
 AND ipca.codigo_hierarquico = inpc.codigo_hierarquico
WHERE ipca.nivel = 'grupo'
GROUP BY ipca.nome_categoria
ORDER BY diferenca_pp DESC;
"""
df_pesos = pd.read_sql_query(query_pesos, con)
df_pesos